In [1]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Update data_path to reflect the mounted Google Drive path
data_path = "/content/drive/My Drive/STAT390 Data/All Calls by Month/"

In [4]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [5]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 64)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)


The code chunk below identifies the columns missing in at least one DataFrame.

In [6]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'Call Recording Result', 'Call Recording Trigger', 'Public Calling IP Address', 'Public Called IP Address', 'PSTN vendor name2', 'User', 'Call Recording Platform Name', 'Redirecting party UUID', 'Column1', 'Device owner UUID', 'External caller ID number'}


The code chunk below prints the columns present in all the data files.

In [7]:
print(common_cols)

{'Authorization code', 'Site timezone', 'Client type', 'Device Mac', 'Related call ID', 'PSTN provider ID', 'Sub client type', 'International Country', 'Report ID', 'Start time', 'Called number', 'Location', 'PSTN vendor name', 'Local SessionID', 'Final remote sessionID', 'Answer time', 'User type', 'Direction', 'Org UUID', 'Inbound trunk', 'PSTN legal entity', 'Route group', 'Remote call ID', 'Call transfer time', 'Report time', 'Outbound trunk', 'Release time', 'Call ID', 'Redirecting number', 'Correlation ID', 'Network call ID', 'User UUID', 'Answer Indicator', 'Call type', 'Remote SessionID', 'Site main number', 'PSTN vendor Org ID', 'Call outcome', 'Site UUID', 'Ring duration', 'Duration', 'Client version', 'Redirect reason', 'Department ID', 'Call outcome reason', 'Model', 'User number', 'OS type', 'Original reason', 'Transfer related call ID', 'Related reason', 'Releasing party', 'Final local sessionID', 'Answered', 'Local call ID'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [8]:
df_main = pd.DataFrame(columns=list(common_cols))

In [9]:
i=0;
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 64)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)


### Converting date to datetime format

In [10]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)

In [11]:
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [12]:
df_main["Start time"].head()

,Start time
0,2024-04-30 18:58:53.988
1,2024-04-30 18:56:37.386
2,2024-04-30 18:54:59.099
3,2024-04-30 18:54:59.099
4,2024-04-30 18:54:52.336


In [13]:
df_allcallsdata = df_main

In [14]:
df_allcallsdata['Direction'].unique()

array(['TERMINATING', 'ORIGINATING'], dtype=object)

### Determine if a call is outbound or inbound

In [15]:
### Remove call duration = 0
# convert the "Duration" column into numeric
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]


### Sort by Correlation ID and Start Time (chronological order)
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])


### Classify call type (Inbound/Outbound/Internal)
def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

df_allcallsdata["Inbound/Outbound"] = df_allcallsdata.apply(classify_call, axis=1)


### Aggregate duration for each call with multiple observations while keeping first record

# Sum total duration per Correlation ID
duration_sum = df_allcallsdata.groupby("Correlation ID")["Duration"].sum().reset_index()
duration_sum.rename(columns={"Duration": "Total Duration"}, inplace=True)

# Keep the first record per Correlation ID (after sorting)
first_rows = df_allcallsdata.groupby("Correlation ID").first().reset_index()

# Merge total duration back to the first-row dataframe
df_merged = first_rows.merge(duration_sum, on="Correlation ID", how="left")


### Export clean dataset
df_merged.to_csv("AllCallsData_Aggregated.csv", index=False)

print(df_merged.head())


                         Correlation ID Authorization code Site timezone  \
0  00000fe9-dfa0-41c5-986b-d9ce731f2715               None          -300   
1  00001e73-ce33-48f1-b531-bddc7ee3965d               None          -300   
2  00006ccc-8250-4993-90c0-bbfd41f7dd24               None          -360   
3  00008bec-c404-48cc-8276-1f276bf4229a               None          -300   
4  000090ae-a71c-49e9-99d6-fdc7078dfa48               None          -300   

        Client type Device Mac Related call ID  \
0              WXCC       None            None   
1              WXCC       None            None   
2               SIP       None   51247234294:0   
3               SIP       None     402835102:0   
4  TEAMS_WXC_CLIENT       None            None   

                       PSTN provider ID Sub client type International Country  \
0  afc59c71-23c9-4884-bab9-535f916eb11b            None                  None   
1  afc59c71-23c9-4884-bab9-535f916eb11b            None                  None   